# setup + imports

In [6]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
from lib import SEED
from lib.paths import ACTS, META
from lib.task import make_task

acts = np.load(ACTS).astype(np.float32)
meta = pd.read_csv(META)

# make_task sets up the required data, namely the prompts and the versions we work with, 
# with the settled upon splits and corresponding labels
task = make_task(acts, meta)

X_train, y_train = task.train
X_test, y_test = task.test
print(X_train.shape, X_test.shape)
print("evading fraction: ", round(y_train.mean(), 3))

(3291, 3584) (822, 3584)
evading fraction:  0.333


# models

In [30]:
# Building the feature set: 

thresholds = np.median(X_train, axis=0)
def mapping(X):
    return (X > thresholds).astype(np.float32)

x_train = mapping(X_train)
x_test = mapping(X_test)

## Naive Bayes

In [31]:
# we use logs so the scale of the numbers we are working with is reasonable
def logbernoulli(x, Px):
    return x @ np.log(Px) + (1 - x) @ np.log(1 - Px)

def fit(X, y):
    PH = y.mean()
    # adding 1 and 2 to prevent values of exactly 0 or 1 with log
    PxH  = (X[y == 1].sum(axis=0) + 1) / ((y == 1).sum() + 2)
    PxHc = (X[y == 0].sum(axis=0) + 1) / ((y == 0).sum() + 2)
    return PH, PxH, PxHc

def calcphx(x, model):
    PH, PxH, PxHc = model
    return 1 / (1 + np.exp((logbernoulli(x, PxHc) + np.log(1 - PH)) - (logbernoulli(x, PxH) + np.log(PH))))

In [32]:
model = fit(x_train, y_train)          
P_train = calcphx(x_train, model)
P_test  = calcphx(x_test, model)

print("baseline (always answer not evading)", round(1 - y_test.mean(), 3))
print("train accuracy", round(((P_train > 0.5) == y_train).mean(), 3))
print("test accuracy", round(((P_test > 0.5) == y_test).mean(), 3))

baseline (always answer not evading) 0.667
train accuracy 0.847
test accuracy 0.856


/var/folders/c9/b70khcxn43d2g3ds9_wfp0yc0000gn/T/ipykernel_58871/2886940774.py:14: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp((logbernoulli(x, PxHc) + np.log(1 - PH)) - (logbernoulli(x, PxH) + np.log(PH))))


## Dependent Bayes

In [42]:
def Pintersect(x):
    n, dim = x.shape
    pi = (x).mean(axis=0) # when P(Xi = 1)
    pij = (x.T @ x) / n # P(Xi = 1 and Xj = 1)
    return pij - np.outer(pi, pi)         


def sortcorrelation(x, y):
    correlations = np.abs(Pintersect(x[y == 1])) + np.abs(Pintersect(x[y == 0]))
    dim = correlations.shape[0]
    order = np.argsort(-correlations.flatten())
    used = np.zeros(dim, dtype=bool)
    pairs = []
    for idx in order:
        i, j = idx//dim, idx%dim
        if i < j and not used[i] and not used[j]:   
            pairs.append((i, j))
            used[i] = True
            used[j] = True
    return np.array(pairs, dtype=int)

In [43]:
def statsperclass(xc, pairs, unpaired):
    n = len(xc)

    pi = np.zeros((len(unpaired), 2))
    for k in range(len(unpaired)):
        i = unpaired[k]
        pi[k, 0] = ((xc[:, i]==0).sum()+1) / (n+2)
        pi[k, 1] = ((xc[:, i]==1).sum()+1) / (n+2)

    pij = np.zeros((len(pairs), 2, 2)) # extra dimension to fit both i and j
    a = xc[:, pairs[:, 0]]     
    b = xc[:, pairs[:, 1]]     
    pij = np.zeros((len(pairs), 2, 2))
    pij[:, 0, 0] = (((a==0)&(b==0)).sum(axis=0)+1) / (n+4)   
    pij[:, 0, 1] = (((a==0)&(b==1)).sum(axis=0)+1) / (n+4)   
    pij[:, 1, 0] = (((a==1)&(b==0)).sum(axis=0)+1) / (n+4)   
    pij[:, 1, 1] = (((a==1)&(b==1)).sum(axis=0)+1) / (n+4)
    return np.log(pi), np.log(pij)

def fit(x, y, pairs):
    x = x.astype(int)
    dim = x.shape[1]
    paired = np.zeros(dim, dtype=bool)
    for i, j in pairs:
        paired[i] = True
        paired[j] = True
    unpaired = np.where(paired == False)[0]
    model = {"pairs": pairs, "unpaired": unpaired}

    x1 = x[y == 1]
    logPH = np.log(len(x1) / len(x))
    logpi, logpij = statsperclass(x1, pairs, unpaired)
    model["H"] = (logPH, logpi, logpij)

    x0 = x[y == 0]
    logPHc = np.log(len(x0) / len(x))
    logpi, logpij = statsperclass(x0, pairs, unpaired)
    model["H^c"] = (logPHc, logpi, logpij)
    return model

def score(x, model, option):
    logPclass, logpi, logpij = model[option]
    unpaired = model["unpaired"]
    pairs = model["pairs"]
    total = np.zeros(len(x))
    total += logPclass
    for k in range(len(unpaired)):
        i = unpaired[k]
        total += logpi[k, x[:, i]]
    for p in range(len(pairs)):
        i, j = pairs[p]
        total += logpij[p, x[:, i], x[:, j]]
    return total   

def bayespredict(x, model):
    x = x.astype(int)
    return score(x, model, "H") - score(x, model, "H^c")


In [45]:
pairs = sortcorrelation(x_train, y_train)

# K = number of pairs we take into account
for K in (0, 250, 500, 1000, len(pairs)):
    model = fit(x_train, y_train, pairs[:K])
    trainacc = ((bayespredict(x_train, model)>0) == y_train).mean()
    testacc = ((bayespredict(x_test, model)>0) == y_test).mean()
    print("K: ", K)
    print("train accuracy: ", round(trainacc, 3))
    print("test accuracy: ", round(testacc, 3))
    print()

K:  0
train accuracy:  0.847
test accuracy:  0.856

K:  250
train accuracy:  0.826
test accuracy:  0.832

K:  500
train accuracy:  0.828
test accuracy:  0.832

K:  1000
train accuracy:  0.847
test accuracy:  0.855

K:  1792
train accuracy:  0.856
test accuracy:  0.867

